# Détection des Glissements de Terrain

Ce notebook utilise les caractéristiques texturales et la pente (DEM) pour identifier les glissements de terrain récents.

In [ ]:
!pip install geemap earthengine-api rasterio opencv-python matplotlib -q
import ee, geemap, rasterio, cv2
import numpy as np
import matplotlib.pyplot as plt

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.1, -4.9, 15.2, -4.8])
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate('2023-01-01', '2023-12-31').median().clip(roi)
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(dem)
geemap.ee_export_image(s2.select(['B4', 'B8']).addBands(slope), 'input.tif', scale=30, region=roi)

In [ ]:
with rasterio.open('input.tif') as src: data = src.read().astype(np.float32)
red, nir, slope = data[0], data[1], data[2]
ndvi = (nir - red) / (nir + red + 1e-8)
edges = cv2.Canny(red.astype(np.uint8), 30, 100) / 255.0

susceptibility = (1.0 - ndvi) * edges * (slope / 90.0)
plt.imshow(susceptibility, cmap='YlOrRd')
plt.title("Carte de Susceptibilité aux Glissements de Terrain")
plt.show()